# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Task 1
# YOUR CODE HERE
# --- Task 1 Solution ---

# 1) Filter continent + years
continent = "Asia"
years = [2000, 2007]

df_t1 = df.loc[
    (df["continent"] == continent) & 
    (df["year"].isin(years))
]

# 2) Scatter plot
fig1 = px.scatter(
    df_t1,
    x="gdpPercap",
    y="lifeExp",
    color="year",
    hover_name="country",
    log_x=True,
    title=f"GDP vs Life Expectancy — {continent} (2000 vs 2007)"
)

# 3) Add arrows showing direction of change for each country
df_2000 = df_t1[df_t1["year"] == 2000].set_index("country")
df_2007 = df_t1[df_t1["year"] == 2007].set_index("country")

for country in df_2000.index:
    fig1.add_annotation(
        x=df_2007.loc[country, "gdpPercap"],
        y=df_2007.loc[country, "lifeExp"],
        ax=df_2000.loc[country, "gdpPercap"],
        ay=df_2000.loc[country, "lifeExp"],
        xref="x", yref="y",
        axref="x", ayref="y",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1
    )

fig1.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [9]:
# Task 2
# YOUR CODE HERE
# --- Task 2 Solution ---

# 1) Filter 2007 data
df_2007 = df[df["year"] == 2007].copy()

highlight_continent = "Europe"

# 2) Create highlight column
df_2007["highlight"] = df_2007["continent"].apply(
    lambda x: highlight_continent if x == highlight_continent else "Other"
)

# 3) Bubble chart
fig2 = px.scatter(
    df_2007,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="highlight",
    log_x=True,
    hover_name="country",
    color_discrete_map={
        highlight_continent: "#1f77b4",
        "Other": "purple"
    },
    title="Global Health vs Wealth (2007)"
)

# 4) Add annotation (storytelling!)
fig2.add_annotation(
    x=30000,
    y=80,
    text="Europe clusters at high wealth<br>and high life expectancy",
    showarrow=True,
    arrowhead=2
)

fig2.show()